# ML-10 — Content Action Playbook

Lane 2 refresh/content opportunity scoring. Decision support for human review, not automated publishing.

## 1. Ranked actions + reason codes

The queue uses the audited ML-09 decision-tree approach and decision-time fields.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
root=Path.cwd()
if not (root/"data/processed/refresh_feature_vector.csv").exists(): root=(root/"../..").resolve()
frame=pd.read_csv(root/"data/processed/refresh_feature_vector.csv")
features=["impressions_90d","days_since_last_update","avg_position","ctr","engagement_rate"]
X=frame[features]; y=frame["is_declining_label"].astype(int)
model=Pipeline([("prep",ColumnTransformer([("num",SimpleImputer(strategy="median"),features)])),("model",DecisionTreeClassifier(class_weight="balanced",max_depth=5,min_samples_leaf=50,random_state=42))])
model.fit(X,y)
q=frame[["content_id","client_id"]+features].copy(); q["action_score"]=model.predict_proba(X)[:,1]
vis=q.impressions_90d.ge(500); lowctr=q.ctr.lt(q.ctr.median()) & vis; stale=q.days_since_last_update.ge(180) & vis; pos=q.avg_position.gt(10) & vis
q["reason_code"]=np.select([stale,lowctr,pos],["STALE_VISIBLE","LOW_CTR_VISIBLE","POSITION_OPPORTUNITY"],default="INSUFFICIENT_EVIDENCE")
q["recommended_action"]=q.reason_code.map({"STALE_VISIBLE":"REFRESH_CONTENT","LOW_CTR_VISIBLE":"REVIEW_TITLE_META","POSITION_OPPORTUNITY":"INVESTIGATE_RANKING","INSUFFICIENT_EVIDENCE":"MANUAL_REVIEW"})
q["evidence_summary"]=q.reason_code.map({"STALE_VISIBLE":"Visible page is at least 180 days old.","LOW_CTR_VISIBLE":"Visible page has below-median CTR.","POSITION_OPPORTUNITY":"Visible page is below page-one position.","INSUFFICIENT_EVIDENCE":"No single documented trigger is sufficient."})
q["confidence_note"]="Ranks association with current decline proxy; not a future forecast."; q["human_review_status"]="REQUIRED"
q=q.sort_values(["action_score","content_id"],ascending=[False,True]).reset_index(drop=True); q["priority_rank"]=np.arange(1,len(q)+1)
print(f"Ranked pages: {len(q):,}; unique pages: {q.content_id.nunique():,}; audited model: decision_tree"); display(q[["priority_rank","content_id","action_score","reason_code","recommended_action","evidence_summary","human_review_status"]].head(10))
assert q.content_id.is_unique and set(features).isdisjoint({"trend_direction","trend_pct","content_id","client_id"})

Ranked pages: 30,000; unique pages: 30,000; audited model: decision_tree
 priority_rank           content_id  action_score          reason_code  recommended_action                         evidence_summary human_review_status
             1 content_00821613c11f      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR.            REQUIRED
             2 content_0255d506d5f2      0.773631 POSITION_OPPORTUNITY INVESTIGATE_RANKING Visible page is below page-one position.            REQUIRED
             3 content_02ae9b37d7b7      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR.            REQUIRED
             4 content_045de7f327fe      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR.            REQUIRED
             5 content_0589f97047f1      0.773631 POSITION_OPPORTUNITY INVESTIGATE_RANKING Visible page is below page-one position.            REQUIRED
             6 

## 2. Intended use and limits

SEO editors and content strategists use this queue to choose pages for review. The score ranks pages associated with the observed current decline proxy in this snapshot; it is not a prospective decline forecast. A high score does not prove a refresh will improve SEO. The ML-09 nested outer result (decision tree Precision@50 0.540 versus baseline 0.280) is directional evidence from one snapshot and an evaluation-specific comparison, not a causal or production guarantee. Recommendations never trigger automatic publishing.

In [2]:
print("Intended use: prioritize human review; no automatic publishing.")
print("ML-09 nested outer holdout: decision tree=0.540; baseline=0.280; client-grouped, one snapshot.")
print("Current decline proxy is not a future outcome; no causal refresh claim.")

Intended use: prioritize human review; no automatic publishing.
ML-09 nested outer holdout: decision tree=0.540; baseline=0.280; client-grouped, one snapshot.
Current decline proxy is not a future outcome; no causal refresh claim.


## 3. Human review + the no-go list

Before action verify the page is live/relevant, metrics are complete, the recommendation matches content, intent and seasonality are understood, cost is justified, and client priorities permit it. Escalate for missing metrics, very low visibility, recent major changes, seasonality, legal/brand approval, contradictory evidence, duplicate/canonicalized pages, or target-derived recommendations. Human review is mandatory.

In [3]:
review_gates=["live/relevant","complete metrics","content/intent","seasonality","cost/value","client approval"]; no_go=["missing critical metrics","very low visibility","recent major change","strong seasonality","legal/brand approval","contradictory evidence","duplicate/canonicalized page","target-derived recommendation"]
print(f"Review gates: {len(review_gates)}; no-go cases: {len(no_go)}"); assert "target-derived recommendation" in no_go

Review gates: 6; no-go cases: 8


## 4. Monitoring / retrain triggers

Monitor data freshness, missingness, score distributions, page/client mix, observed decline rates, reviewer acceptance/rejection, post-action performance, and evaluation deterioration. Investigate shifts; revalidate with a new client-grouped evaluation when population or performance changes; retrain only after timing and label windows are re-established; suspend recommendations if critical data are stale, missing, or leakage is suspected. Cost/value is directional: LOW title/meta review, MEDIUM content refresh, HIGH ranking/strategy investigation. No monetary ROI is available.

In [4]:
triggers=["data freshness","missing-feature rate","score distribution","page/client mix","decline rate","acceptance/rejection","post-action performance","evaluation deterioration"]; print("Monitoring triggers:",", ".join(triggers)); print("Prioritize evidence strength and opportunity against editorial effort; no monetary ROI measured.")

Monitoring triggers: data freshness, missing-feature rate, score distribution, page/client mix, decline rate, acceptance/rejection, post-action performance, evaluation deterioration
Prioritize evidence strength and opportunity against editorial effort; no monetary ROI measured.


## 5. Exports for the paper

The CSV is regenerated from authorized starter-derived features. Generated CSVs remain ignored and are not committed.

In [5]:
out=root/"work/outputs/content_action_playbook.csv"; out.parent.mkdir(parents=True,exist_ok=True)
cols=["priority_rank","content_id","action_score","reason_code","recommended_action","evidence_summary","confidence_note","human_review_status"]; q[cols].to_csv(out,index=False)
print(f"Wrote {out} with {len(q):,} rows and {len(cols)} columns."); print(q[cols].head(10).to_string(index=False)); assert out.exists() and q[cols].reason_code.notna().all()

Wrote C:\Users\John Ray\Desktop\flyrank-ml-internship-sinagdev\work\outputs\content_action_playbook.csv with 30,000 rows and 8 columns.
 priority_rank           content_id  action_score          reason_code  recommended_action                         evidence_summary                                                      confidence_note human_review_status
             1 content_00821613c11f      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR. Ranks association with current decline proxy; not a future forecast.            REQUIRED
             2 content_0255d506d5f2      0.773631 POSITION_OPPORTUNITY INVESTIGATE_RANKING Visible page is below page-one position. Ranks association with current decline proxy; not a future forecast.            REQUIRED
             3 content_02ae9b37d7b7      0.773631      LOW_CTR_VISIBLE   REVIEW_TITLE_META       Visible page has below-median CTR. Ranks association with current decline proxy; not a future forecast. 

## Self-check

- [x] Audited ML-09 decision-tree setup and decision-time fields used.
- [x] Reason codes, actions, evidence, uncertainty, and mandatory human review are explicit.
- [x] Intended use, limitations, no-go cases, cost/value, and monitoring are documented.
- [x] Current decline is a proxy, not a future forecast; no causal claim.
- [x] CSV is reproducible and ignored.
- [x] IDs and target-derived fields are excluded from features.